# 06 — Python Module Search Path

## Learning objectives

In this notebook, you will learn how Python searches for modules, how to inspect `sys.path`, why the current working directory differs from a script's directory, how Python environments and `PYTHONPATH` affect imports, and how to troubleshoot common import errors.

Examples that depend on a particular folder structure are shown as snippets with filenames. Their output can vary by Python version, operating system, and how Python is launched.

## 01. Introduction to module search paths

When Python encounters an import such as `import example`, it must locate a module or package matching that name. The locations Python searches are represented by `sys.path` and the import system.

A **module name** is the name used in an import statement, such as `example`. A **file path** identifies a location on disk, such as `C:/work/example.py`. They are related, but they are not interchangeable: writing a file path as an ordinary module name does not tell Python to search that exact path.

Python's search locations depend on how the interpreter was started, its environment, and its configuration.

In [ ]:
import sys

print("Module search path entries:")
for path in sys.path:
    print(path)

Review the output from your own Python environment. Common entries include a script or working directory, standard-library locations, and directories containing installed third-party packages. The exact list and order vary.

The first entry is often the script directory when running a script, or the current directory in some interactive modes. Do not assume it has the same meaning in every execution mode.

## 02. Understanding `sys.path`

`sys.path` is a Python list of strings representing locations used when searching for modules. Python considers the configured locations in order, so a module in an earlier location can take precedence over a same-named module in a later location.

Inspect it with:

```python
import sys
for path in sys.path:
    print(path)
```

To check whether a particular directory is present, compare its resolved path with the entries. Relative paths and empty-string entries can represent the current directory, so inspect them in the context of how Python was launched.

In [ ]:
from pathlib import Path
import sys

target_dir = Path.cwd().resolve()
resolved_entries = []
for entry in sys.path:
    # An empty entry commonly represents the current working directory.
    candidate = Path(entry or ".").resolve()
    resolved_entries.append(candidate)

print("Current directory:", target_dir)
print("Current directory represented in sys.path:",
      target_dir in resolved_entries)

Avoid editing `sys.path` without a clear reason. Search-path order affects which module Python finds, and adding a directory near the beginning can shadow a standard-library or third-party module.

## 03. The current working directory

Two useful values are:

- `os.getcwd()` — the process's current working directory.
- `sys.path[0]` — the first search-path entry, whose meaning depends on the execution mode.

They are not guaranteed to be equal. When running a script, the first entry commonly points to the script's directory. When using an interactive interpreter, notebook, or IDE, path setup can differ.

This explains why a relative file path or import can work from one launch location and fail from another.

In [ ]:
import os
import sys

print("Working directory:", os.getcwd())
print("First search-path entry:", repr(sys.path[0] if sys.path else None))

For example, if a script is stored in `C:/work/app/main.py` but the process was started with `C:/work` as its working directory, the working directory and script directory are different concepts. The exact `sys.path` values depend on how the script was launched.

## 04. Python environments and installed modules

Python commonly loads imports from several categories:

- **Standard library:** modules distributed with Python, such as `math`, `pathlib`, and `statistics`.
- **Third-party packages:** packages installed separately, such as `pandas`.
- **`site-packages`:** a common directory where third-party packages are installed for a particular Python installation or environment.

A package installed in one environment may not be available in another. For example, an IDE can use a virtual environment while a terminal command uses a different Python interpreter.

Useful diagnostic commands in a terminal are:

```text
python --version
python -m pip --version
python -m pip show pandas
```

Using `python -m pip` helps associate pip with the interpreter named by `python`. On Windows, `py -0p` can also list registered Python installations.

In [ ]:
import sys
import site

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("User site-packages:", site.getusersitepackages())
print("Site-package directories:")
for directory in site.getsitepackages():
    print(" ", directory)

If a package is reported as missing, compare the interpreter used to install it with the interpreter running the program. Do not assume that installing a package globally makes it available in every virtual environment.

## 05. The `PYTHONPATH` environment variable

`PYTHONPATH` is an environment variable that can add directories to Python's module search path. It is optional; Python can work normally when it is unset.

Inspect it with:

```python
import os
print(os.environ.get("PYTHONPATH", "PYTHONPATH is not set"))
```

Be deliberate when setting it. Broad or persistent additions can make imports depend on machine-specific configuration and can cause name conflicts. For many applications, a well-organized package and a correctly selected environment are easier to maintain.

In [ ]:
import os

pythonpath = os.environ.get("PYTHONPATH")
if pythonpath is None:
    print("PYTHONPATH is not set")
else:
    print("PYTHONPATH:", pythonpath)
    print("Configured entries:")
    for entry in pythonpath.split(os.pathsep):
        print(" ", entry)

## 06. Modifying `sys.path`

For a short experiment, you can add a directory to `sys.path`. Suppose this structure exists:

```text
project/
├── main.py
└── utilities/
    └── calculator.py
```

`utilities/calculator.py`:

```python
def add(a, b):
    return a + b
```

If you run `main.py` from the `project` directory, you can temporarily add `utilities` to the path:

```python
import sys
from pathlib import Path

utilities_dir = str((Path(__file__).resolve().parent / "utilities"))
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)

import calculator
print(calculator.add(10, 20))
```

Expected output:

```text
30
```

This snippet is intended for a `.py` file because `__file__` is generally defined for scripts and imported modules, but not in every interactive environment. In a notebook, use an explicitly chosen directory path instead.

For a larger codebase, prefer a clear package structure and appropriate project configuration over repeated manual path edits.

A package-style alternative is:

```text
project/
├── main.py
└── utilities/
    ├── __init__.py
    └── calculator.py
```

When the project directory is available on the import path, `main.py` can use:

```python
from utilities.calculator import add
print(add(10, 20))
```

This introduces package imports, which are covered in more detail in the upcoming package notebooks.

## 07. Debugging import errors

`ModuleNotFoundError` means Python could not find a module requested by an import. Work through these checks:

1. **Check the spelling.** Module names are case-sensitive on some platforms and should match the actual module name.
2. **Check the file or package location.** Confirm that the intended directory is on the search path.
3. **Check the interpreter.** Inspect `sys.executable` and compare it with the interpreter used to install the package.
4. **Check the environment.** Confirm that the required third-party package is installed in that environment.
5. **Check name conflicts.** A local file such as `random.py`, `statistics.py`, or `pandas.py` can shadow the intended module.
6. **Check the launch method.** A script run from a terminal, an IDE, and a notebook may have different working directories and search paths.
7. **Inspect what was imported.** For a module that imports successfully, `module.__file__` often shows the file Python loaded.

### Example: diagnosing an unexpected import

If a file named `random.py` is created in your working directory, `import random` may load that local file instead of the standard-library module. Rename the local file to a more specific name and remove any stale `__pycache__` files if needed, then restart the interpreter and try again.

For an installed package that cannot be found, compare:

```text
python -m pip show pandas
python -c "import sys; print(sys.executable)"
```

Run these with the same `python` command used to launch the program. If the IDE uses another interpreter, select the intended environment in the IDE.

In [ ]:
import importlib
import sys

print("Interpreter:", sys.executable)
for module_name in ("math", "pathlib"):
    module = importlib.import_module(module_name)
    print(f"{module_name!r} imported from:", getattr(module, "__file__", "<built-in or frozen>"))

## Key takeaways

- Python uses its import system and configured search locations to find modules.
- `sys.path` exposes the search-path entries used by the interpreter.
- The current working directory and script directory are different concepts.
- Standard-library modules and third-party packages may come from different locations.
- `PYTHONPATH` can add search directories, but it should be configured carefully.
- Manual changes to `sys.path` can help with experiments, but package structure and environment configuration are usually easier to maintain.
- To diagnose `ModuleNotFoundError`, check the name, location, interpreter, environment, and possible filename conflicts.